# Engineering diagram classifier — Azure Content Understanding

Classifies an input image (PNG / JPEG / TIFF / BMP / PDF page) into one label:
**SingleLineDiagram**, **ControllerLogic** or **Other** — without returning the full extraction.

Two approaches are compared side by side:

| | Option A — `classify` field | Option B — `contentCategories` classifier |
|---|---|---|
| How | Custom analyzer on `prebuilt-document` with one enum field, `method: classify` | Analyzer with `contentCategories` + `enableSegment: false` |
| Output | `fields.diagramType` = label (+ confidence) | `segments[].category` |
| Best for | "Just give me the label" | Later routing each category to its own extraction analyzer (`analyzerId`) |

> **Why not `prebuilt-image`?** The image base analyzer only supports `returnDetails` — `contentCategories`,
> `enableSegment` and `omitContent` are ignored there. Images are sent to a **document** analyzer instead
> (document analyzers accept image files).

**Setup:** copy `.env.sample` to `.env`, fill it in, then `pip install -r requirements.txt`.
Uses the native [`azure-ai-contentunderstanding`](https://pypi.org/project/azure-ai-contentunderstanding/) SDK.

In [ ]:
import json, mimetypes, os, time
from pathlib import Path

from azure.ai.contentunderstanding import ContentUnderstandingClient
from azure.ai.contentunderstanding.models import AnalysisInput
from azure.core.credentials import AzureKeyCredential
from azure.identity import DefaultAzureCredential
from dotenv import load_dotenv

load_dotenv()

ENDPOINT     = os.environ["CONTENTUNDERSTANDING_ENDPOINT"].rstrip("/")
API_KEY      = os.getenv("CONTENTUNDERSTANDING_KEY")          # optional – falls back to Entra ID
API_VERSION  = os.getenv("CU_API_VERSION", "2025-11-01")       # GA. Try 2026-06-01-preview for layout-aware classification
COMPLETION_MODEL = os.getenv("CU_COMPLETION_MODEL", "gpt-5.2")  # model NAME (not deployment name)
SAMPLES_DIR  = Path(os.getenv("SAMPLES_DIR", "./samples"))

ANALYZER_A = os.getenv("CU_ANALYZER_A", "diagramTypeFieldClassifier")
ANALYZER_B = os.getenv("CU_ANALYZER_B", "diagramTypeCategoryClassifier")

print("Endpoint:", ENDPOINT, "| API:", API_VERSION, "| model:", COMPLETION_MODEL)

## Client
Uses the API key if set, otherwise `DefaultAzureCredential` (az login / managed identity). The SDK handles auth, api-version and long-running-operation polling.

In [ ]:
credential = AzureKeyCredential(API_KEY) if API_KEY else DefaultAzureCredential()
client = ContentUnderstandingClient(ENDPOINT, credential, api_version=API_VERSION)

## (Optional, one-off) Set resource-level default model deployments
Only needed if you haven't done this already in Content Understanding Studio. Maps model **names** to your **deployment** names.

In [ ]:
SET_DEFAULTS = False   # flip to True once

if SET_DEFAULTS:
    defaults = client.update_defaults(model_deployments={
        os.environ["CU_COMPLETION_MODEL"]: os.environ["CU_COMPLETION_DEPLOYMENT"],
        os.environ["CU_EMBEDDING_MODEL"]:  os.environ["CU_EMBEDDING_DEPLOYMENT"],
    })
    print(defaults.model_deployments)

## Category definitions — the main quality lever
Describe each class by **what the drawing is and what it communicates** (its subject and standard conventions), not by traits of particular test images — that overfits. Keep an explicit `Other` so unrelated images aren't forced into a class. Validate any change on a labelled set (`samples/<Label>/`), never on a single image.

In [ ]:
CATEGORIES = {
    "SingleLineDiagram": (
        "Electrical power-system drawing that shows how electrical power is distributed and connected: the topology "
        "from supply sources through buses, switching and protection devices (breakers, disconnects, fuses, relays), "
        "transformers, reactors, filters, drives and starters, to loads such as motors. Depicts power equipment as "
        "symbols joined by conductors, annotated with ratings (voltage, current, kVA/MVA, frequency), equipment tags "
        "and protection device numbers. May use one-line or multi-line (e.g. three-phase) notation. "
        "The subject is the power circuit and its equipment."
    ),
    "ControllerLogic": (
        "Control or automation drawing that shows how signals and logic determine equipment behavior: PLC ladder logic "
        "(rails, rungs, contacts, coils), function block diagrams, logic gate or sequence diagrams, interlock and "
        "permissive logic, cause-and-effect matrices, and control-system I/O or tag documentation. "
        "The subject is signal flow and logic (inputs, conditions, outputs), not the power equipment itself."
    ),
    "Other": (
        "Any content that is not primarily an electrical power-system drawing or a control-logic drawing: photos, "
        "piping and instrumentation diagrams (P&IDs), architectural or civil plans, layouts, mechanical drawings, "
        "datasheets, tables, text documents, charts, or unrelated images."
    ),
}
LABELS = list(CATEGORIES)

## Option A — single `classify` field (label only)
`omitContent` must stay `false` when a field schema is defined (the service rejects `true`); `enableFigureDescription` gives the model a caption of the
drawing, which helps on text-sparse diagrams.

In [ ]:
raw = {}   # file -> cached analysis results; cleared whenever an analyzer is (re)created

def create_analyzer(analyzer_id, definition):
    client.begin_create_analyzer(analyzer_id, definition, allow_replace=True).result()
    raw.clear()
    print("Created analyzer:", analyzer_id)

enum_desc = " ".join(f"{k}: {v}" for k, v in CATEGORIES.items())

analyzer_a = {
    "baseAnalyzerId": "prebuilt-document",
    "description": "Classifies engineering drawings/images by diagram type. Returns a single label only.",
    "config": {
        "returnDetails": False,
        "omitContent": False,
        "enableOcr": True,
        "enableLayout": True,
        "enableFormula": False,
        "enableBarcode": False,
        "enableFigureDescription": True,
        "estimateFieldSourceAndConfidence": True,
    },
    "fieldSchema": {
        "name": "DiagramType",
        "fields": {
            "diagramType": {
                "type": "string",
                "method": "classify",
                "enum": LABELS,
                "description": "The type of diagram shown in the input. " + enum_desc,
            }
        },
    },
    "models": {"completion": COMPLETION_MODEL},
}

create_analyzer(ANALYZER_A, analyzer_a)

## Option B — `contentCategories` classifier
`enableSegment: false` classifies the whole file into exactly one category. No `analyzerId` per category → classification only.

> Keep `omitContent: false` here — the `segments` array (which carries the category) lives on the original content object.
> To keep the response small, `enableFigureDescription`/`returnDetails` stay off; you just read `segments[].category`.
> Later, add `"analyzerId": "<extractor>"` to a category to route it for extraction.

In [ ]:
analyzer_b = {
    "baseAnalyzerId": "prebuilt-document",
    "description": "Classifies engineering drawings/images by diagram type.",
    "config": {
        "returnDetails": False,
        "enableSegment": False,
        "omitContent": False,
        "enableOcr": True,
        "enableLayout": True,
        "contentCategories": {name: {"description": desc} for name, desc in CATEGORIES.items()},
        # Example of routing later:
        # "contentCategories": {"SingleLineDiagram": {"description": "...", "analyzerId": "mySldEquipmentExtractor"}, ...}
    },
    "models": {"completion": COMPLETION_MODEL},
}

create_analyzer(ANALYZER_B, analyzer_b)

## Analyze helper
Sends local files as binary. For files already in Blob storage use `analyze_url`.

In [ ]:
def analyze_file(analyzer_id, path: Path):
    mime = mimetypes.guess_type(path.name)[0] or "application/octet-stream"
    return client.begin_analyze_binary(analyzer_id, path.read_bytes(), content_type=mime).result()

def analyze_url(analyzer_id, url: str):
    return client.begin_analyze(analyzer_id, inputs=[AnalysisInput(url=url)]).result()

## Result parsers — pull only the label

In [ ]:
def label_from_field(result):
    """Option A: fields.diagramType"""
    for c in result.contents or []:
        f = (c.fields or {}).get("diagramType")
        if f:
            return f.value, f.confidence
    return None, None

def label_from_categories(result):
    """Option B: segments[].category (or top-level category on a routed content object)"""
    for c in result.contents or []:
        segs = getattr(c, "segments", None)
        if segs:
            return segs[0].category, None
        if c.category:
            return c.category, None
    return None, None

## Try multiple images
Runs both options on the images in `SAMPLES_DIR` (recursively), in parallel. Set `SAMPLE_LIMIT` to a number to only try the first N files, or `None` for all.
Raw responses are cached in `raw` (cleared when an analyzer is re-created), so re-running a cell doesn't re-analyze files. Re-run this cell after adding or moving images.

In [ ]:
from concurrent.futures import ThreadPoolExecutor
import pandas as pd

SAMPLE_LIMIT = None   # e.g. 5 to try only the first 5 files
MAX_WORKERS  = 4

IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp", ".pdf", ".heif"}

def list_samples():
    return sorted(p for p in SAMPLES_DIR.rglob("*") if p.suffix.lower() in IMAGE_EXTS)

sample_files = list_samples()
print(f"{len(sample_files)} files found in {SAMPLES_DIR}")

def classify_file(f: Path):
    """Run both options on one file (cached) and return a result row."""
    row = {"file": str(f.relative_to(SAMPLES_DIR)),
           "expected": f.parent.name if f.parent.name in LABELS else None}
    try:
        if f not in raw:
            t0 = time.time(); ra = analyze_file(ANALYZER_A, f); a_sec = round(time.time() - t0, 1)
            t0 = time.time(); rb = analyze_file(ANALYZER_B, f); b_sec = round(time.time() - t0, 1)
            raw[f] = {"A": ra, "B": rb, "A_sec": a_sec, "B_sec": b_sec}
        ra, rb = raw[f]["A"], raw[f]["B"]
        row["A_sec"], row["B_sec"] = raw[f]["A_sec"], raw[f]["B_sec"]
        row["A_label"], row["A_conf"] = label_from_field(ra)
        row["B_label"], row["B_conf"] = label_from_categories(rb)
    except Exception as e:
        row["error"] = str(e)[:300]
    return row

def classify_files(files):
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
        return pd.DataFrame(list(pool.map(classify_file, files)))

df = classify_files(sample_files[:SAMPLE_LIMIT])
display(df)

Inspect the raw response of one file if a label comes back empty (change the index to pick another file):

In [ ]:
if raw:
    f = list(raw)[0]
    print(f.name)
    print(json.dumps(raw[f]["A"].as_dict(), indent=2, default=str)[:4000])

## Batch run + accuracy
Put images in sub-folders named after the expected label to get an accuracy score, e.g.
`samples/SingleLineDiagram/*.png`, `samples/ControllerLogic/*.png`, `samples/Other/*.jpg`.

In [ ]:
sample_files = list_samples()   # re-scan so new/moved files are picked up
print(f"{len(sample_files)} files found in {SAMPLES_DIR}")
df = classify_files(sample_files)   # reuses cached results from above
display(df)

labelled = df[df["expected"].notna()]
if len(labelled):
    for opt in ("A", "B"):
        acc = (labelled[f"{opt}_label"] == labelled["expected"]).mean()
        print(f"Option {opt} accuracy: {acc:.1%}  (n={len(labelled)})")
    print(pd.crosstab(labelled["expected"], labelled["A_label"], rownames=["expected"], colnames=["Option A"]))
    print(pd.crosstab(labelled["expected"], labelled["B_label"], rownames=["expected"], colnames=["Option B"]))

Save results:

In [ ]:
df.to_csv("classification_results.csv", index=False)
print("Saved classification_results.csv")

## A/B the preview API (optional)
`2026-06-01-preview` adds layout-based classification signals (figure descriptions, section markers) that help on
text-sparse / figure-heavy pages — exactly the diagram case. Set `CU_API_VERSION=2026-06-01-preview` in `.env`,
restart the kernel, re-run, and compare the accuracy numbers. Preview has no SLA — use GA for production.

## Clean up

In [ ]:
CLEANUP = False
if CLEANUP:
    for a in (ANALYZER_A, ANALYZER_B):
        client.delete_analyzer(a)
        print("Deleted", a)